# 03 — Modeling

Treinamento, avaliação e interpretabilidade dos modelos:
- Baseline (regressão linear)
- XGBoost vs LightGBM (cross-validation)
- SHAP values — feature importance
- Análise de erros

In [ ]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import xgboost as xgb
import lightgbm as lgb
from pathlib import Path
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

PROCESSED_PATH = Path('../data/processed')

## 1. Carregamento do Dataset

In [ ]:
from src.training.train import _load_features, _rmse_original_space, _mae_original_space

X, y = _load_features()
print(f'X shape: {X.shape}')
print(f'y (log_price) — mean: {y.mean():.2f}, std: {y.std():.2f}')
print(f'y (price) — mediana: R${np.expm1(y).median():.2f}')

## 2. Baseline — Ridge Regression

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

baseline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0))
])

oof_baseline = np.zeros(len(y))
for train_idx, val_idx in kf.split(X):
    baseline.fit(X.iloc[train_idx], y.iloc[train_idx])
    oof_baseline[val_idx] = baseline.predict(X.iloc[val_idx])

rmse_baseline = _rmse_original_space(y.values, oof_baseline)
mae_baseline = _mae_original_space(y.values, oof_baseline)
print(f'Baseline Ridge — RMSE: R${rmse_baseline:.2f} | MAE: R${mae_baseline:.2f}')

## 3. XGBoost

In [ ]:
from src.training.train import XGBOOST_PARAMS

oof_xgb = np.zeros(len(y))
xgb_models = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = xgb.XGBRegressor(**XGBOOST_PARAMS)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = model.predict(X_val)
    xgb_models.append(model)
    print(f'Fold {fold+1}: RMSE=R${_rmse_original_space(y_val.values, oof_xgb[val_idx]):.2f}')

rmse_xgb = _rmse_original_space(y.values, oof_xgb)
mae_xgb = _mae_original_space(y.values, oof_xgb)
print(f'\nXGBoost OOF — RMSE: R${rmse_xgb:.2f} | MAE: R${mae_xgb:.2f}')
print(f'Melhoria vs Baseline: {(rmse_baseline - rmse_xgb) / rmse_baseline * 100:.1f}%')

## 4. LightGBM

In [ ]:
from src.training.train import LIGHTGBM_PARAMS

oof_lgb = np.zeros(len(y))
lgb_models = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = lgb.LGBMRegressor(**LIGHTGBM_PARAMS)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)])
    oof_lgb[val_idx] = model.predict(X_val)
    lgb_models.append(model)

rmse_lgb = _rmse_original_space(y.values, oof_lgb)
mae_lgb = _mae_original_space(y.values, oof_lgb)
print(f'LightGBM OOF — RMSE: R${rmse_lgb:.2f} | MAE: R${mae_lgb:.2f}')

In [ ]:
# Comparação dos modelos
results = pd.DataFrame({
    'Modelo': ['Ridge Baseline', 'XGBoost', 'LightGBM'],
    'RMSE (R$)': [rmse_baseline, rmse_xgb, rmse_lgb],
    'MAE (R$)': [mae_baseline, mae_xgb, mae_lgb],
})
results = results.sort_values('RMSE (R$)')

fig, axes = plt.subplots(1, 2)
results.plot(x='Modelo', y='RMSE (R$)', kind='bar', ax=axes[0], color='steelblue', legend=False)
axes[0].set_title('RMSE por Modelo')
axes[0].set_xlabel('')
axes[0].set_ylabel('R$')
axes[0].tick_params(axis='x', rotation=15)

results.plot(x='Modelo', y='MAE (R$)', kind='bar', ax=axes[1], color='teal', legend=False)
axes[1].set_title('MAE por Modelo')
axes[1].set_xlabel('')
axes[1].set_ylabel('R$')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()
print(results.to_string(index=False))

## 5. SHAP — Feature Importance

In [ ]:
# Treinar modelo final no dataset completo para SHAP
best_params = XGBOOST_PARAMS.copy()
best_params.pop('early_stopping_rounds', None)
final_model = xgb.XGBRegressor(**best_params)
final_model.fit(X, y)

# SHAP values (amostra de 500 para velocidade)
sample_idx = np.random.choice(len(X), min(500, len(X)), replace=False)
X_sample = X.iloc[sample_idx]

explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_sample)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, max_display=20, show=False)
plt.title('SHAP — Top 20 Features mais importantes')
plt.tight_layout()
plt.savefig(PROCESSED_PATH / 'shap_summary.png', bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
# SHAP Bar plot — importância global
mean_shap = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=X.columns
).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 5))
mean_shap.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Importância Global das Features (|SHAP| médio)')
ax.set_xlabel('|SHAP value| médio')
plt.tight_layout()
plt.show()

## 6. Análise de Erros

In [ ]:
# Usar OOF predictions do melhor modelo
best_oof = oof_xgb if rmse_xgb <= rmse_lgb else oof_lgb

errors = pd.DataFrame({
    'y_true': np.expm1(y.values),
    'y_pred': np.expm1(best_oof),
})
errors['error'] = errors['y_pred'] - errors['y_true']
errors['abs_error'] = errors['error'].abs()
errors['pct_error'] = errors['error'] / errors['y_true'] * 100

fig, axes = plt.subplots(1, 2)

# Predicted vs True
axes[0].scatter(errors['y_true'], errors['y_pred'], alpha=0.3, s=5, color='steelblue')
lim = [0, errors[['y_true', 'y_pred']].max().max()]
axes[0].plot(lim, lim, 'r--', linewidth=1)
axes[0].set_xlabel('Preço Real (R$)')
axes[0].set_ylabel('Preço Previsto (R$)')
axes[0].set_title('Previsto vs Real')

# Distribuição dos erros
axes[1].hist(errors['pct_error'].clip(-100, 100), bins=60, color='teal', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Erro percentual (%)')
axes[1].set_title(f'Distribuição de Erros (μ={errors["pct_error"].mean():.1f}%)')

plt.tight_layout()
plt.show()

print(f'Erros > 50%: {(errors["abs_error"] / errors["y_true"] > 0.5).sum()} listings')
print(f'Erros > 100%: {(errors["abs_error"] / errors["y_true"] > 1.0).sum()} listings')

In [ ]:
# Piores previsões — investigar
worst = errors.nlargest(10, 'abs_error')
print('Top 10 maiores erros absolutos:')
print(worst[['y_true', 'y_pred', 'error', 'pct_error']].to_string())